# 06 业务成本收益与敏感性分析

- 同一口径：凡涉及"筛选减少注册/试听"，一律写 **点估计与方向三法一致、显著性强度依赖 click 相互独立假设（day-cluster 口径下 Gross CI 跨 0）**，不做无条件显著的表述。
- 所有业务成本/工时/毛利/外推流量参数均为 **Assumption — not observed in dataset**，与观测值严格区分；成本收益整体属于 **scenario analysis**。
- 效应区间双口径并列：**iid-Z 口径**与 **day-cluster bootstrap 口径**（以天为 cluster，Gross CI 含 0）；本步的保守下界必须取 cluster 口径（≈0 甚至为负那端），不得用 iid 下界充当保守值。
- 数值全部沿用/5 已锁定结果，不重算、不改动。

In [1]:
# 加载主分析的 Gross 效应（iid CI + day-cluster bootstrap CI）与业务假设
import json, tomllib
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
CFG = tomllib.load(open(ROOT/"config"/"analysis_config.toml","rb"))
main = json.loads((ROOT/"data"/"processed"/"main_effects.json").read_text(encoding="utf-8"))
g = main["z_tests"]["GrossConversion"]; gb = main["bootstrap"]["GrossConversion"]
d = g["abs_diff"]
iid_lo, iid_hi = g["ci95_low"], g["ci95_high"]          # iid-Z 95%CI
clu_lo, clu_hi = gb["ci95_low"], gb["ci95_high"]        # day-cluster bootstrap 95%CI
tot = {r["Group"]: r for r in main["totals"]}
n_e_out = int(tot["Experiment"]["Clicks"])
BA = CFG["business_assumptions"]
print("Gross point=%.6f" % d)
print("iid-Z CI=(%.6f, %.6f) | day-cluster bootstrap CI=(%.6f, %.6f)" % (iid_lo,iid_hi,clu_lo,clu_hi))
print("业务假设标签：", BA["label"], "| 货币：", BA["currency"])

Gross point=-0.020555
iid-Z CI=(-0.029120, -0.011990) | day-cluster bootstrap CI=(-0.045871, 0.004770)
业务假设标签： Assumption - not observed in dataset; scenario analysis only | 货币： USD


## 被筛除的免费试听规模（iid 与 day-cluster 双口径并列）
反事实：减少量 = 实验组 clicks ×（p_C−p_E）= −n×d；区间 = −n×效应 CI。
- **iid-Z 口径**：区间为正但依赖 click 相互独立假设；
- **day-cluster bootstrap 口径**：承认日间异质性，Gross CI 含 0 → 减少量**下界为负（即不排除反而增加）**，这才是保守口径。

In [2]:
def reduction_range(n, lo, hi, point=d):
    return -n*point, -n*hi, -n*lo   # point, low, high

rows = []
for tag, n in [("observed outcome window (23d)", n_e_out),
               ("37d traffic window (SCENARIO: lag days same effect)", 28325)]:
    pt_iid, lo_iid, hi_iid = reduction_range(n, iid_lo, iid_hi)
    pt_clu, lo_clu, hi_clu = reduction_range(n, clu_lo, clu_hi)
    rows.append({"scale": tag, "clicks": n,
                 "iid_point": pt_iid, "iid_low": lo_iid, "iid_high": hi_iid,
                 "cluster_point": pt_clu, "cluster_low": lo_clu, "cluster_high": hi_clu})
scale = pd.DataFrame(rows)
print(scale.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
print("\n每万 clicks：iid %.1f(%.1f~%.1f)；cluster %.1f(%.1f~%.1f)"
      % (-d*1e4, -iid_hi*1e4, -iid_lo*1e4, -d*1e4, -clu_hi*1e4, -clu_lo*1e4))
print("观测窗日均减少（point）≈ %.1f 例/天；相对减少 %.2f%%" % (-n_e_out*d/23, abs(g["relative_change"]*100)))

                                              scale  clicks  iid_point  iid_low  iid_high  cluster_point  cluster_low  cluster_high
                      observed outcome window (23d)   17260      354.8    206.9     502.6          354.8        -82.3         791.7
37d traffic window (SCENARIO: lag days same effect)   28325      582.2    339.6     824.8          582.2       -135.1       1,299.3

每万 clicks：iid 205.5(119.9~291.2)；cluster 205.5(-47.7~458.7)
观测窗日均减少（point）≈ 15.4 例/天；相对减少 9.39%


## 业务参数（情景假设，全部为 Assumption — not observed in dataset）
- 单试听用户支持工时：5 / 15 / 30 分钟；综合时薪：15 / 30 / 60 USD/小时 → **单试听支持成本 c = 1.25 / 7.5 / 30 USD**。
- 单付费用户贡献毛利 v = 50 / 150 / 400 USD，**仅用于 break-even**。
- 年化外推：每组 1,600 clicks/天（=设计基线 3,200/2，本身为假设）×365；headline 永远用 23 天观测窗。
- 以上均非数据集观测，任何输出都带 Assumption 标注。

In [3]:
c_low, c_base, c_high = BA["cost_per_enrollment_low_base_high"]
v_low, v_base, v_high = BA["payment_margin_low_base_high"]
# 交叉核对 c = 分钟×时薪/60
mins = BA["support_minutes_low_base_high"]; rates = BA["hourly_loaded_cost_low_base_high"]
calc = [m*r/60 for m, r in zip(mins, rates)]
assert np.allclose(calc, BA["cost_per_enrollment_low_base_high"])
print("单试听成本交叉核对 分钟×时薪/60 =", calc, "== config", BA["cost_per_enrollment_low_base_high"])
print("成本三档 c =", BA["cost_per_enrollment_low_base_high"], "USD；毛利三档 v =",
      BA["payment_margin_low_base_high"], "USD（仅 break-even 用）")

单试听成本交叉核对 分钟×时薪/60 = [1.25, 7.5, 30.0] == config [1.25, 7.5, 30]
成本三档 c = [1.25, 7.5, 30] USD；毛利三档 v = [50, 150, 400] USD（仅 break-even 用）


## Low/Base/High 成本情景 × 效应三档（保守下界取 cluster 口径）
效应三档：**cluster 下界（≈0/负，保守）/ point / cluster 上界**；iid 区间仅作参照列。规模用 23 天观测窗（headline）。节约 = 减少量 × c。

In [4]:
# 观测窗三档效应（cluster 为保守口径）
pt = -n_e_out*d
eff = {"cluster_lower (conservative)": -n_e_out*clu_hi,
       "point": pt,
       "cluster_upper": -n_e_out*clu_lo}
iid_ref = (-n_e_out*iid_hi, -n_e_out*iid_lo)
scen_rows = []
for cname, c in zip(["Low (c=1.25)","Base (c=7.5)","High (c=30)"], [c_low,c_base,c_high]):
    scen_rows.append({"cost_scenario": cname, "cost_per_enrollment_USD": c,
                      "savings_cluster_lower": eff["cluster_lower (conservative)"]*c,
                      "savings_point": eff["point"]*c,
                      "savings_cluster_upper": eff["cluster_upper"]*c,
                      "iid_reference_range": f"[{iid_ref[0]*c:,.0f}, {iid_ref[1]*c:,.0f}]"})
scen = pd.DataFrame(scen_rows)
print("23天观测窗资源节约（USD，Assumption 成本 × 反事实减少量；负=不排除净增投入）")
print(scen.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
print("\n效应三档减少量（例）：cluster 下界 %.1f / point %.1f / cluster 上界 %.1f；iid 参照 [%0.1f, %0.1f]"
      % (eff["cluster_lower (conservative)"], eff["point"], eff["cluster_upper"], iid_ref[0], iid_ref[1]))

23天观测窗资源节约（USD，Assumption 成本 × 反事实减少量；负=不排除净增投入）
cost_scenario  cost_per_enrollment_USD  savings_cluster_lower  savings_point  savings_cluster_upper iid_reference_range
 Low (c=1.25)                      1.2                 -102.9          443.5                  989.7          [259, 628]
 Base (c=7.5)                      7.5                 -617.4        2,660.8                5,938.0      [1,552, 3,770]
  High (c=30)                     30.0               -2,469.7       10,643.3               23,752.1     [6,208, 15,078]

效应三档减少量（例）：cluster 下界 -82.3 / point 354.8 / cluster 上界 791.7；iid 参照 [206.9, 502.6]


In [5]:
# 落盘结果
out = {"screened_scale": scale.to_dict(orient="records"),
       "assumptions": {**BA},
       "scenarios_23d": scen_rows,
       "effect_tiers_reduction_23d": eff, "iid_reference_reduction_23d": list(iid_ref),
       "unified_wording": "point/direction agree across Z/delta/bootstrap; strength depends on iid-click assumption; day-cluster Gross CI contains 0"}
p = ROOT/"data"/"processed"/"cost_benefit.json"
p.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
back = json.loads(p.read_text(encoding="utf-8"))
assert abs(back["effect_tiers_reduction_23d"]["point"]-pt) < 1e-9
print("cost_benefit.json (steps 37-39) written & re-read OK")

cost_benefit.json (steps 37-39) written & re-read OK


## 敏感性：cost × reduction 网格、tornado、break-even

In [6]:
# (1) cost × 效应档位 网格（23天观测窗，USD；负值=不排除净增投入）
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
cost_grid = np.array([1.25, 2.5, 5, 7.5, 10, 15, 20, 30])
tiers = {"cluster_lower": eff["cluster_lower (conservative)"],
         "point": eff["point"], "cluster_upper": eff["cluster_upper"]}
grid_df = pd.DataFrame({k: np.round(cost_grid*red, 1) for k, red in tiers.items()},
                       index=pd.Index(cost_grid, name="cost_USD_per_enrollment"))
print(grid_df.to_string())

fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=150)
im = ax.imshow(grid_df.to_numpy(), aspect="auto", cmap="RdYlGn",
               vmin=-3000, vmax=24000)
ax.set_xticks(range(3), ["cluster lower", "point", "cluster upper"])
ax.set_yticks(range(len(cost_grid)), [f"{c:.2f}" for c in cost_grid])
ax.set_xlabel("Gross effect tier (day-cluster conservative)"); ax.set_ylabel("Assumed cost per enrollment (USD)")
for i in range(grid_df.shape[0]):
    for j in range(grid_df.shape[1]):
        ax.text(j, i, f"{grid_df.iloc[i,j]:,.0f}", ha="center", va="center", fontsize=7)
ax.set_title("23d operational savings (USD): cost x effect grid [Assumption]")
fig.colorbar(im, ax=ax, label="USD", shrink=.8)
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_savings_grid.png"); plt.close(fig)
print("saved fig_savings_grid.png")

                         cluster_lower    point  cluster_upper
cost_USD_per_enrollment                                       
1.25                            -102.9    443.5          989.7
2.50                            -205.8    886.9         1979.3
5.00                            -411.6   1773.9         3958.7
7.50                            -617.4   2660.8         5938.0
10.00                           -823.2   3547.8         7917.4
15.00                          -1234.8   5321.7        11876.1
20.00                          -1646.5   7095.5        15834.7
30.00                          -2469.7  10643.3        23752.1
saved fig_savings_grid.png


In [7]:
# (2) Tornado：以 Base(c=7.5, point 效应) 为中心，单因素摆动
base_saving = eff["point"]*c_base
swing_effect = (eff["cluster_lower (conservative)"]*c_base, eff["cluster_upper"]*c_base)
swing_cost = (eff["point"]*c_low, eff["point"]*c_high)
iid_swing = (iid_ref[0]*c_base, iid_ref[1]*c_base)
tor = pd.DataFrame([
    ["unit cost c (1.25..30)", *swing_cost],
    ["Gross effect (cluster CI)", *swing_effect],
    ["Gross effect (iid CI, ref.)", *iid_swing]], columns=["driver","low","high"])
print(tor.to_string(index=False, float_format=lambda x: f"{x:,.1f}")); print("base =", f"{base_saving:,.1f}")

fig, ax = plt.subplots(figsize=(8, 3.6), dpi=150)
ys = range(len(tor))
for y, (_, row) in zip(ys, tor.iterrows()):
    ax.barh(y, row["high"]-base_saving, left=base_saving, color="#2ca02c", alpha=.75)
    ax.barh(y, row["low"]-base_saving, left=base_saving, color="#d62728", alpha=.75)
ax.axvline(base_saving, color="black", lw=1.3, label=f"Base={base_saving:,.0f}")
ax.set_yticks(list(ys), tor["driver"]); ax.invert_yaxis()
ax.set_xlabel("23d operational savings (USD) [Assumption]")
ax.set_title("Tornado: drivers of savings uncertainty (23d window)")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=.3)
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_tornado.png"); plt.close(fig)
print("saved fig_tornado.png")

                     driver     low     high
     unit cost c (1.25..30)   443.5 10,643.3
  Gross effect (cluster CI)  -617.4  5,938.0
Gross effect (iid CI, ref.) 1,552.1  3,769.6
base = 2,660.8
saved fig_tornado.png


In [8]:
# (3) Break-even（SCENARIO；依赖未观测的 v 与外推）
net = main["z_tests"]["NetConversion"]
lost_payments_pt = -net["abs_diff"]*n_e_out          # Net 点估对应的付费用户数变化（23d 实验组）
red_pt = eff["point"]
be_rows = []
for vname, v in zip(["Low v=50","Base v=150","High v=400"], [v_low,v_base,v_high]):
    loss_pt = lost_payments_pt*v
    c_star = loss_pt/red_pt                          # 节约=损失时的临界单试听成本
    be_rows.append({"margin_scenario": vname, "v_USD": v,
                    "point_revenue_loss_USD": loss_pt,
                    "breakeven_cost_per_enrollment_USD": c_star})
be = pd.DataFrame(be_rows)
print("点估付费减少 = %.2f 例（23d）；运营节约点估 = %.1f USD（Base c=7.5）" % (lost_payments_pt, base_saving))
print(be.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
v_star_base = base_saving/lost_payments_pt
print("Base 成本下的临界毛利 v* = %.2f USD/付费：真实毛利低于它则点估节约 > 点估收入下行" % v_star_base)
print("Net 95%%CI 对应付费变化区间 [%0.1f, %0.1f] 例（含增益可能），break-even 随之更不确定，仅作 scenario"
      % (-net["ci95_high"]*n_e_out, -net["ci95_low"]*n_e_out))

# 年化情景行（明确外推假设）
apd = BA["annualization_clicks_per_group_per_day"]*BA["annualization_days"]
ann_red = -d*apd; ann_pay = -net["abs_diff"]*apd
print("年化 SCENARIO（每组1600 clicks/天×365=584,000，Assumption）：减少试听 %.0f 例；节约 "
      "%.0f/%.0f/%.0f USD；点估付费减少 %.0f 例" % (ann_red, ann_red*c_low, ann_red*c_base, ann_red*c_high, ann_pay))

fig, ax = plt.subplots(figsize=(7.6, 4.2), dpi=150)
xs = np.linspace(0, 35, 200)
ax.plot(xs, red_pt*xs, color="#1f4e79", lw=2, label="Operational savings (point)")
for v, col in zip([v_low,v_base,v_high], ["#2ca02c","#e69f00","#d62728"]):
    ax.axhline(lost_payments_pt*v, ls="--", color=col, lw=1.3,
               label=f"Revenue downside v={v} (break-even c*={lost_payments_pt*v/red_pt:.1f})")
ax.set_xlabel("Assumed cost per screened enrollment (USD) [Assumption]")
ax.set_ylabel("USD over 23d outcome window")
ax.set_title("Break-even: operational savings vs Net revenue downside [SCENARIO]")
ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(ROOT/"reports"/"figures"/"fig_breakeven.png"); plt.close(fig)
print("saved fig_breakeven.png")

点估付费减少 = 84.12 例（23d）；运营节约点估 = 2660.8 USD（Base c=7.5）
margin_scenario  v_USD  point_revenue_loss_USD  breakeven_cost_per_enrollment_USD
       Low v=50     50                4,206.02                              11.86
     Base v=150    150               12,618.07                              35.57
     High v=400    400               33,648.18                              94.84
Base 成本下的临界毛利 v* = 31.63 USD/付费：真实毛利低于它则点估节约 > 点估收入下行
Net 95%CI 对应付费变化区间 [-32.0, 200.3] 例（含增益可能），break-even 随之更不确定，仅作 scenario
年化 SCENARIO（每组1600 clicks/天×365=584,000，Assumption）：减少试听 12004 例；节约 15005/90030/360121 USD；点估付费减少 2846 例


saved fig_breakeven.png


## 外推条件披露
1. 观测窗数字为反事实估计；37 天窗、年化均为 **scenario analysis，不是观测结果**，须假设 treatment effect 可迁移、未来流量结构（渠道/设备/星期）与实验期相似、无组间溢出、无新奇效应。
2. 所有 c/v、1,600 clicks/组/天均为 **Assumption — not observed in dataset**；break-even 同时依赖未观测的贡献毛利 v 与效应点估，**不是观测结论**。
3. 同一口径不变：点估计与方向三法一致、显著性强度依赖 click 独立假设（day-cluster Gross CI 含 0），因此节约区间的保守端为负。

## 成本收益业务小结
- **转化效果**：筛选使免费试听注册在 Z 主分析下下降 9.39%（−2.06pp）；同一口径——点估与方向三法一致、显著性强度依赖 click 独立假设，day-cluster 口径 CI 含 0。
- **非劣效风险**：最终付费 Net 点估 −0.49pp，未通过 δ=0.75pp 非劣效门（下界 −1.16pp），属精度不足的"未能确立"，不是已证劣效。
- **资源节约（23 天观测窗，Assumption 成本）**：点估减少 355 例试听；Base 成本（7.5 USD/例）下节约 ≈2,661 USD，但 day-cluster 保守区间为 [−617, +5,938] USD——**节约方向与点估成立，保守端不排除净增投入**。
- **break-even（scenario）**：点估口径下临界单试听成本 c*=11.9/35.6/94.9 USD（对应 v=50/150/400）；只有当真实支持成本高于对应 c* 时，运营节约才盖过点估收入下行。
- 业务含义：策略"省资源"的点估故事成立，但与 Net 非劣效未通过一样，**当前证据强度不足以支撑无条件全量上线**，最终建议在后续分析 综合给出。

In [9]:
# 补落盘结果
out = json.loads(p.read_text(encoding="utf-8"))
out["sensitivity"] = {"savings_grid": grid_df.reset_index().to_dict(orient="records"),
                 "tornado_base": base_saving, "tornado": tor.to_dict(orient="records"),
                 "breakeven": be.to_dict(orient="records"),
                 "breakeven_margin_at_base_cost": float(v_star_base),
                 "point_lost_payments_23d": float(lost_payments_pt),
                 "annualized_scenario": {"clicks_per_group": int(apd),
                                         "reduced_enrollments": float(ann_red),
                                         "savings_low_base_high": [float(ann_red*c_low),float(ann_red*c_base),float(ann_red*c_high)],
                                         "point_lost_payments": float(ann_pay),
                                         "label": "Assumption - not observed in dataset"}}
p.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
back = json.loads(p.read_text(encoding="utf-8"))
assert abs(back["sensitivity"]["tornado_base"]-base_saving) < 1e-9
print("cost_benefit.json completed (steps 37-42) & re-read OK")

cost_benefit.json completed (steps 37-42) & re-read OK
